# 预训练数据混合：从语料库存到可审计采样配方

**面试问题：多领域预训练语料怎样设置 mixture、temperature、上下界和反馈更新？**

## 回答主线

先明确业务目标和数据合同，再给出可比较的朴素基线；随后手写核心算法，展示中间状态、最终指标和失败路径。本 Notebook 的断言只出现在最后，用于保护关键不变量；学习重点是前面的输入、过程、对照与解释。

## 真实案例

一家中文客服模型团队有通用网页、商品知识、客服对话、代码文档和合规制度五个语料域。原始 token 量相差两百倍，质量与验证损失也不同。目标是在 1 亿 token 的续训预算内提升客服与合规能力，同时避免小域被重复几十轮、通用能力明显退化。下面的数据是脱敏后的真实字段结构，数值为教学缩小版。

### 输入预览：先看库存、质量与目标域

In [1]:
from pprint import pprint  # 导入结构化打印工具，便于直接观察领域语料字段。
domains = [  # 构造具有真实训练配方字段的五个语料域。
    {"name": "通用网页", "tokens_m": 8200, "quality": 0.72, "val_loss": 2.42, "target_loss": 2.45, "min_share": 0.25, "max_share": 0.55},  # 通用域负责保持基础语言能力。
    {"name": "商品知识", "tokens_m": 900, "quality": 0.86, "val_loss": 2.18, "target_loss": 2.05, "min_share": 0.12, "max_share": 0.30},  # 商品域是客服回答的主要事实来源。
    {"name": "客服对话", "tokens_m": 260, "quality": 0.91, "val_loss": 2.36, "target_loss": 1.95, "min_share": 0.18, "max_share": 0.32},  # 客服域是本轮续训的重点能力域。
    {"name": "代码文档", "tokens_m": 1200, "quality": 0.78, "val_loss": 2.09, "target_loss": 2.15, "min_share": 0.05, "max_share": 0.18},  # 代码域需要保留但不应挤占客服预算。
    {"name": "合规制度", "tokens_m": 40, "quality": 0.95, "val_loss": 2.71, "target_loss": 2.10, "min_share": 0.08, "max_share": 0.12},  # 合规域很小，因此必须设置最大重复上限。
]  # 完成领域配置列表。
print("五个语料域的原始配置：")  # 输出标题帮助学习者理解随后表格的语义。
pprint(domains, sort_dicts=False)  # 展示每个域的库存、质量、损失和上下界。

五个语料域的原始配置：
[{'name': '通用网页',
  'tokens_m': 8200,
  'quality': 0.72,
  'val_loss': 2.42,
  'target_loss': 2.45,
  'min_share': 0.25,
  'max_share': 0.55},
 {'name': '商品知识',
  'tokens_m': 900,
  'quality': 0.86,
  'val_loss': 2.18,
  'target_loss': 2.05,
  'min_share': 0.12,
  'max_share': 0.3},
 {'name': '客服对话',
  'tokens_m': 260,
  'quality': 0.91,
  'val_loss': 2.36,
  'target_loss': 1.95,
  'min_share': 0.18,
  'max_share': 0.32},
 {'name': '代码文档',
  'tokens_m': 1200,
  'quality': 0.78,
  'val_loss': 2.09,
  'target_loss': 2.15,
  'min_share': 0.05,
  'max_share': 0.18},
 {'name': '合规制度',
  'tokens_m': 40,
  'quality': 0.95,
  'val_loss': 2.71,
  'target_loss': 2.1,
  'min_share': 0.08,
  'max_share': 0.12}]


## Baseline 基线：按原始 token 量等比例采样

In [2]:
def normalize(weights):  # 定义把任意正权重归一化为概率的基础函数。
    total = sum(weights)  # 计算所有领域权重的总和作为归一化分母。
    return [weight / total for weight in weights]  # 返回和为一的领域采样概率。

proportional = normalize([row["tokens_m"] for row in domains])  # 计算最常见的按库存比例采样基线。
print("按库存比例采样的结果：")  # 输出基线标题，便于和后续策略对比。
for row, share in zip(domains, proportional):  # 逐域展示基线概率和目标差距。
    print(f'{row["name"]:<8} share={share:6.2%} 目标损失差={row["val_loss"] - row["target_loss"]:+.2f}')  # 显示大域垄断预算以及困难域被忽略的问题。

按库存比例采样的结果：
通用网页     share=77.36% 目标损失差=-0.03
商品知识     share= 8.49% 目标损失差=+0.13
客服对话     share= 2.45% 目标损失差=+0.41
代码文档     share=11.32% 目标损失差=-0.06
合规制度     share= 0.38% 目标损失差=+0.61


### 核心实现：temperature 平滑与上下界投影

In [3]:
def bounded_temperature_mix(rows, alpha=0.45, rounds=30):  # 实现带上下界的 temperature 领域混合算法。
    raw = [row["tokens_m"] ** alpha * row["quality"] for row in rows]  # 用幂次平滑库存并乘入质量权重。
    shares = normalize(raw)  # 得到尚未满足业务约束的初始概率。
    for _ in range(rounds):  # 迭代投影以同时满足每个域的最小和最大份额。
        clipped = [min(max(share, row["min_share"]), row["max_share"]) for share, row in zip(shares, rows)]  # 把每个域裁剪到允许区间。
        shares = normalize(clipped)  # 重新归一化使所有概率之和保持为一。
    return shares  # 返回可直接进入采样器的领域概率。

temperature_mix = bounded_temperature_mix(domains)  # 对五个真实语义领域计算平滑后的采样配方。
print("temperature + 质量 + 上下界后的配方：")  # 输出改进策略标题。
for row, before, after in zip(domains, proportional, temperature_mix):  # 对比每个域调整前后的概率。
    print(f'{row["name"]:<8} 比例基线={before:6.2%}  调整后={after:6.2%}  变化={after - before:+6.2%}')  # 展示小域得到预算且大域仍保留底座。

temperature + 质量 + 上下界后的配方：
通用网页     比例基线=77.36%  调整后=39.93%  变化=-37.43%
商品知识     比例基线= 8.49%  调整后=17.65%  变化=+9.15%
客服对话     比例基线= 2.45%  调整后=18.00%  变化=+15.55%
代码文档     比例基线=11.32%  调整后=16.43%  变化=+5.11%
合规制度     比例基线= 0.38%  调整后= 8.00%  变化=+7.62%


### 反馈更新：用 excess loss 调整下一轮预算

In [4]:
import math  # 导入指数更新需要的数学函数。

def feedback_update(current, rows, eta=0.8):  # 实现一轮 DoReMi 风格的 excess-loss 反馈更新。
    excess = [max(row["val_loss"] - row["target_loss"], 0.0) for row in rows]  # 只对尚未达到目标的领域计算正损失缺口。
    tilted = [share * math.exp(eta * gap) for share, gap in zip(current, excess)]  # 按损失缺口指数放大困难领域权重。
    proposal = normalize(tilted)  # 把乘法权重转换回概率分布。
    projected_rows = [dict(row, tokens_m=proposal[index] ** (1 / 0.45) / row["quality"]) for index, row in enumerate(rows)]  # 构造等价权重以复用边界投影器。
    return bounded_temperature_mix(projected_rows, alpha=0.45)  # 返回满足业务上下界的反馈配方。

feedback_mix = feedback_update(temperature_mix, domains)  # 根据当前验证损失生成下一轮 mixture。
print("加入验证损失反馈后的下一轮配方：")  # 输出反馈更新结果标题。
for row, share in zip(domains, feedback_mix):  # 逐域查看反馈后的预算分配。
    print(f'{row["name"]:<8} share={share:6.2%}  excess={max(row["val_loss"] - row["target_loss"], 0.0):.2f}')  # 展示高损失域获得更多预算但仍受上限约束。

加入验证损失反馈后的下一轮配方：
通用网页     share=32.80%  excess=0.00
商品知识     share=17.74%  excess=0.13
客服对话     share=23.35%  excess=0.41
代码文档     share=14.11%  excess=0.00
合规制度     share=12.00%  excess=0.61


## 结果解读：预算、重复轮数与可审计 manifest

In [5]:
budget_m = 100  # 设定本轮教学实验的一亿 token 训练预算。
allocation = [budget_m * share for share in feedback_mix]  # 把概率转换成各域实际 token 配额。
print("最终预算与等效重复轮数：")  # 输出容量规划表标题。
for row, tokens in zip(domains, allocation):  # 逐域计算预算和数据重复程度。
    epochs = tokens / row["tokens_m"]  # 用分配 token 除以去重库存估算等效 epoch。
    print(f'{row["name"]:<8} budget={tokens:6.2f}M  equivalent_epochs={epochs:6.3f}')  # 显示小域重复风险和大域覆盖比例。
manifest = {row["name"]: {"share": round(share, 4), "budget_m": round(tokens, 2)} for row, share, tokens in zip(domains, feedback_mix, allocation)}  # 生成可版本化的训练配方清单。
print("可写入训练制品的 mixture manifest：", manifest)  # 展示发布时需要固化的精确概率与预算。

最终预算与等效重复轮数：
通用网页     budget= 32.80M  equivalent_epochs= 0.004
商品知识     budget= 17.74M  equivalent_epochs= 0.020
客服对话     budget= 23.35M  equivalent_epochs= 0.090
代码文档     budget= 14.11M  equivalent_epochs= 0.012
合规制度     budget= 12.00M  equivalent_epochs= 0.300
可写入训练制品的 mixture manifest： {'通用网页': {'share': 0.328, 'budget_m': 32.8}, '商品知识': {'share': 0.1774, 'budget_m': 17.74}, '客服对话': {'share': 0.2335, 'budget_m': 23.35}, '代码文档': {'share': 0.1411, 'budget_m': 14.11}, '合规制度': {'share': 0.12, 'budget_m': 12.0}}


## 失败案例：去掉小域上限会发生什么

In [6]:
uncapped_rows = [dict(row, max_share=0.95) for row in domains]  # 构造没有实际最大份额保护的错误配置。
uncapped = feedback_update(temperature_mix, uncapped_rows, eta=3.0)  # 用过强反馈模拟小域验证噪声放大。
bad_compliance_share = uncapped[-1]  # 读取只有四千万 token 库存的合规域份额。
good_compliance_share = feedback_mix[-1]  # 读取经过上限保护的合规域份额。
print(f"错误配置：合规域份额={bad_compliance_share:.2%}，等效重复={budget_m * bad_compliance_share / domains[-1]['tokens_m']:.2f} epoch")  # 展示噪声反馈造成的过采样。
print(f"修正配置：合规域份额={good_compliance_share:.2%}，等效重复={budget_m * good_compliance_share / domains[-1]['tokens_m']:.2f} epoch")  # 展示上限如何控制记忆和过拟合风险。

错误配置：合规域份额=25.03%，等效重复=0.63 epoch
修正配置：合规域份额=12.00%，等效重复=0.30 epoch


### 生产边界

In [7]:
release_gate = {"sum_to_one": abs(sum(feedback_mix) - 1.0) < 1e-9, "within_bounds": all(row["min_share"] - 0.01 <= share <= row["max_share"] + 0.01 for row, share in zip(domains, feedback_mix)), "max_small_domain_epochs": max(tokens / row["tokens_m"] for row, tokens in zip(domains, allocation))}  # 汇总训练发布前需要审计的关键门禁。
print("发布门禁摘要：", release_gate)  # 输出概率合法性、约束与最大重复轮数。
print("生产替换点：真实系统还需接入去重库存、许可、污染检测、在线 loss 估计和跨轮 checkpoint 版本。")  # 明确教学 mixture 与生产数据平台之间的边界。

发布门禁摘要： {'sum_to_one': True, 'within_bounds': True, 'max_small_domain_epochs': 0.3}
生产替换点：真实系统还需接入去重库存、许可、污染检测、在线 loss 估计和跨轮 checkpoint 版本。


## 回归测试：只保护关键不变量

In [8]:
assert abs(sum(feedback_mix) - 1.0) < 1e-9  # 验证最终 mixture 是合法概率分布。
assert feedback_mix[2] > proportional[2]  # 验证重点客服域相对库存基线获得更多训练预算。
assert feedback_mix[-1] <= domains[-1]["max_share"] + 0.01  # 验证小型合规域没有突破配置上限。
assert len(manifest) == len(domains)  # 验证发布 manifest 覆盖全部领域且没有漏项。
print("回归测试通过：概率、重点域、上限和 manifest 四个核心合同均成立。")  # 用可见消息说明测试范围而非隐藏在大量断言中。

回归测试通过：概率、重点域、上限和 manifest 四个核心合同均成立。
